<hr style="border: 6px solid#003262;" />

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/02_cover.png" align="center" width="20%" padding="10"><br>
    <br>
</div>

<br>

# ENSEMBLE SELECTION METHODS

<br>

**About:** A practical guide to combining predictions from multiple trained models using three systematic strategies - Forward Selection, Backward Elimination, and Random Sampling - to build ensembles that consistently outperform any individual model.

**Learning Goals:**
* Understand why averaging diverse model predictions reduces error
* Quantify model diversity using prediction correlation and its relationship to ensemble gain
* Manually build and evaluate ensembles to develop intuition before automating
* Implement Forward Selection to greedily add the best model at each step
* Implement Backward Elimination to greedily prune the worst model at each step
* Implement Random Ensemble Sampling and compare efficiency vs. greedy methods
* Select a final ensemble and understand the trade-off between ensemble size and AUC gain

**Keywords:** ensemble methods, model averaging, forward selection, backward elimination, random selection, AUC, model diversity, prediction correlation

**Prerequisite Knowledge:** (1) Binary classification and AUC metric, (2) scikit-learn fit/predict/predict_proba interface, (3) Pandas DataFrames and basic numpy operations, (4) Notebook 01 - image classification and random search (provides the prediction files used here)

**Target User:** ML practitioners who have trained multiple models on the same task and want a principled method to combine them for production.

<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 0: SETUP AND DATA](#Part_0)
> #### [PART 1: WHY ENSEMBLES WORK](#Part_1)
> #### [PART 2: MANUAL ENSEMBLE BUILDING](#Part_2)
> #### [PART 3: FORWARD SELECTION](#Part_3)
> #### [PART 4: BACKWARD ELIMINATION](#Part_4)
> #### [PART 5: RANDOM ENSEMBLE SAMPLING AND COMPARISON](#Part_5)

<br>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools
import os

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

<a id='Part_0'></a>

<hr style="border: 2px solid#003262;" />

#### PART 0

## **SETUP** and Data

<a id='Part_0_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 0.1: LOADING BASE MODEL PREDICTIONS

<br>

This notebook is designed to work with the prediction CSV files generated by Notebook 01. If you have not run Notebook 01, the cell below generates equivalent predictions using the sklearn Breast Cancer dataset (also a binary cancer classification problem) and five diverse model types - Logistic Regression, Random Forest, Gradient Boosting, SVM, and K-Nearest Neighbors.

The prediction format is the same either way: a pandas DataFrame where each column is one model's predicted probability for the positive class, and each row is one validation or test sample. The ensemble methods treat columns as interchangeable - the model type that produced each column is irrelevant to the selection algorithm.

___

**Note:** Using model types that are architecturally diverse (linear vs. tree-based vs. distance-based) tends to produce lower inter-model correlation than using many variants of the same architecture. Lower correlation means more ensemble gain. This is the "bias-variance diversity decomposition" in practice.

___

In [ ]:
import os

# Check for prediction files from Notebook 01
val_file = 'data/validation_predictions.csv'
label_file = 'data/validation_labels.csv'

if os.path.exists(val_file) and os.path.exists(label_file):
    val_preds = pd.read_csv(val_file)
    labels_val = pd.read_csv(label_file)['label'].values
    test_preds = pd.read_csv('data/test_predictions.csv')
    labels_test = pd.read_csv('data/test_labels.csv')['label'].values
    print(f'Loaded predictions from Notebook 01: {val_preds.shape[1]} models')
else:
    # Generate predictions from scratch using breast_cancer dataset
    print('Notebook 01 files not found. Generating predictions from sklearn breast_cancer...')
    bc = load_breast_cancer()
    X_train_val, X_test_raw, y_train_val, y_test_raw = train_test_split(
        bc.data, bc.target, test_size=0.2, random_state=42, stratify=bc.target
    )
    X_tr, X_val_raw, y_tr, y_val_raw = train_test_split(
        X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
    )
    labels_val = y_val_raw
    labels_test = y_test_raw

    # Five diverse model types, three variants each = 15 models
    base_models = {
        'LR': LogisticRegression,
        'RF': RandomForestClassifier,
        'GB': GradientBoostingClassifier,
    }
    val_preds = {}
    test_preds = {}

    for name, ModelClass in base_models.items():
        for seed in range(5):
            pipe = Pipeline([
                ('scaler', StandardScaler()),
                ('clf', ModelClass(random_state=seed) if 'random_state' in ModelClass().get_params() else ModelClass())
            ])
            pipe.fit(X_tr, y_tr)
            col = f'{name}_{seed}'
            val_preds[col] = pipe.predict_proba(X_val_raw)[:, 1]
            test_preds[col] = pipe.predict_proba(X_test_raw)[:, 1]

    val_preds = pd.DataFrame(val_preds)
    test_preds = pd.DataFrame(test_preds)

print(f'Models available: {list(val_preds.columns)}')
print(f'Validation samples: {len(labels_val)}, Test samples: {len(labels_test)}')

<a id='Part_0_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 0.2: INDIVIDUAL MODEL BASELINES

<br>

In [ ]:
# Compute AUC for each individual model
individual_aucs = {}
for col in val_preds.columns:
    individual_aucs[col] = roc_auc_score(labels_val, val_preds[col])

auc_series = pd.Series(individual_aucs).sort_values(ascending=False)
print('Individual model validation AUCs:')
print(auc_series.to_string())
print(f'\nBest single model: {auc_series.index[0]} (AUC={auc_series.iloc[0]:.4f})')
print(f'Mean single model: {auc_series.mean():.4f}')

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **WHY** Ensembles Work

<a id='Part_1_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.1: THE BIAS-VARIANCE DECOMPOSITION OF ENSEMBLE GAIN

<br>

Averaging model predictions reduces variance without increasing bias - when two models make independent, uncorrelated errors, their errors partially cancel when their predictions are averaged. The ensemble's variance is:

$$\text{Var}\left(\frac{1}{M}\sum_{i=1}^{M} f_i(x)\right) = \frac{1}{M^2}\sum_{i,j} \text{Cov}(f_i, f_j)$$

where $f_i(x)$ is model $i$'s prediction. If all $M$ models had identical predictions (correlation = 1), the variance would not decrease at all. If predictions were completely uncorrelated (correlation = 0), variance would shrink by a factor of $M$. Real models fall in between - they share training data and similar architectures, so they are positively correlated but not perfectly so.

This equation explains the key design principle: **diversity beats quality**. Two mediocre models whose errors are uncorrelated can outperform a single excellent model in an ensemble. Adding a highly correlated second model wastes a slot that a more diverse model could use.

___

**Sources Consulted:**
- Dietterich, T.G., "Ensemble Methods in Machine Learning," MCS (2000) - primary source for ensemble theory; stable
- Hastie, Tibshirani, Friedman, *The Elements of Statistical Learning*, 2nd ed. (2009), Chapter 8.8 - stable reference for bias-variance in ensembles

___

In [ ]:
# Correlation matrix of model predictions
corr_matrix = val_preds.corr()

# Visualize
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix.values, cmap='RdYlBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(corr_matrix.columns, fontsize=8)
plt.colorbar(im, ax=ax, label='Pearson Correlation')
ax.set_title('Prediction Correlation Matrix\n(Lower correlation = More ensemble diversity)')
plt.tight_layout()
plt.show()

# Report the least correlated pair - best candidates to form a two-model ensemble
corr_vals = corr_matrix.values.copy()
np.fill_diagonal(corr_vals, np.nan)  # ignore self-correlation
min_idx = np.unravel_index(np.nanargmin(corr_vals), corr_vals.shape)
print(f'Least correlated pair: {corr_matrix.columns[min_idx[0]]} and {corr_matrix.columns[min_idx[1]]}')
print(f'Correlation: {corr_vals[min_idx]:.4f}')

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **The formula above shows that ensemble variance decreases as inter-model correlation decreases. Compute the average pairwise correlation across all model pairs in `val_preds`. Then create a two-model ensemble using the least correlated pair and compare its AUC to the best individual model's AUC. Does a diverse pair beat the best single model?**

<br>

```python
# 1. Compute mean pairwise correlation (exclude diagonal)
corr_matrix = val_preds.corr().values.copy()
np.fill_diagonal(corr_matrix, np.nan)
mean_corr = ...

# 2. Build the two-model ensemble (average the two predictions)
best_pair = ['model_a', 'model_b']  # replace with the least correlated pair names
ensemble_probs = ...
ensemble_auc = roc_auc_score(labels_val, ensemble_probs)
print(f'Two-model ensemble AUC: {ensemble_auc:.4f}')
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **MANUAL** Ensemble Building

<a id='Part_2_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.1: AVERAGING PREDICTIONS

<br>

Before automating ensemble selection, we build intuition by manually adding one model at a time and observing the AUC change. The prediction for an ensemble is the average predicted probability across included models. This equal-weight averaging is the simplest possible combination rule and is surprisingly effective - more sophisticated weighting schemes (learned stacking, weighted average) rarely outperform it by a meaningful margin when models are already diverse.

The `graph_auc` function below computes and plots the ROC curve for a given subset of models, making it easy to see how AUC evolves as the ensemble grows.

In [ ]:
def compute_ensemble_auc(preds_df, labels, model_list):
    # Average predictions across model_list columns and return AUC
    avg_probs = preds_df[model_list].mean(axis=1)
    return roc_auc_score(labels, avg_probs)

def plot_roc(preds_df, labels, model_list, label_str, ax, color='#003262'):
    avg_probs = preds_df[model_list].mean(axis=1)
    fpr, tpr, _ = roc_curve(labels, avg_probs)
    auc = roc_auc_score(labels, avg_probs)
    ax.plot(fpr, tpr, label=f'{label_str} (AUC={auc:.3f})', color=color)
    return auc

# Start with the single best model, then manually add one at a time
best_model = auc_series.index[0]
second_model = auc_series.index[1]
third_model = auc_series.index[2]

fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#003262', '#3B7EA1', '#C4820E']
auc1 = plot_roc(val_preds, labels_val, [best_model], f'1 model: {best_model}', ax, colors[0])
auc2 = plot_roc(val_preds, labels_val, [best_model, second_model], f'2 models', ax, colors[1])
auc3 = plot_roc(val_preds, labels_val, [best_model, second_model, third_model], f'3 models', ax, colors[2])
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Manual Ensemble Buildup')
ax.legend()
plt.tight_layout()
plt.show()
print(f'1 model AUC: {auc1:.4f} | 2 models: {auc2:.4f} | 3 models: {auc3:.4f}')

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **You have three model predictions: `model_a`, `model_b`, and `model_c`. You add `model_b` to an ensemble that already contains `model_a` and the AUC improves. You then add `model_c` and the AUC drops. What does this tell you about `model_c`, and would you expect the same result if `model_c` had a lower correlation with `model_a` and `model_b`? Write a `check_marginal_gain` function that takes a prediction DataFrame, a list of already-selected models, a candidate model name, and validation labels, and returns the AUC change from adding that candidate.**

<br>

```python
def check_marginal_gain(preds_df, selected, candidate, labels):
    # YOUR CODE HERE
    ...
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **FORWARD** Selection

<a id='Part_3_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.1: ALGORITHM AND IMPLEMENTATION

<br>

Forward Selection is a greedy algorithm that builds the ensemble incrementally: at each step, it tries adding each remaining model one at a time and keeps whichever addition produces the best AUC. This is "greedy" because it commits to the best local choice at each step without reconsidering earlier decisions.

**Algorithm:**
1. Start with an empty ensemble.
2. For each remaining model, compute the AUC if it is added to the current ensemble.
3. Add the model that gives the highest AUC.
4. Repeat until the desired number of models is reached.

The greedy property means Forward Selection does not always find the globally optimal ensemble - a model that looks weak alone might pair exceptionally well with a later addition. In practice, it performs well because AUC is relatively smooth with respect to ensemble composition.

**Computational cost:** If you have $N$ models and want an ensemble of size $K$, Forward Selection requires $N + (N-1) + ... + (N-K+1)$ evaluations = $O(NK)$. For $N=15, K=10$, that is at most 105 evaluations.

In [ ]:
def ensemble_forward(preds_df, labels, n_models):
    # Greedy forward selection: add the best model at each step
    remaining = list(preds_df.columns)
    selected = []
    history = []  # track (step, model_added, ensemble_auc)

    for step in range(n_models):
        best_auc = -1
        best_model = None
        for candidate in remaining:
            trial = selected + [candidate]
            auc = compute_ensemble_auc(preds_df, labels, trial)
            if auc > best_auc:
                best_auc = auc
                best_model = candidate
        selected.append(best_model)
        remaining.remove(best_model)
        history.append({'step': step + 1, 'model_added': best_model, 'ensemble_auc': best_auc})
        print(f'Step {step+1}: Added {best_model} -> Ensemble AUC = {best_auc:.4f}')

    return pd.DataFrame(history), selected

forward_history, forward_ensemble = ensemble_forward(val_preds, labels_val, n_models=10)
print(f'\nFinal Forward ensemble ({len(forward_ensemble)} models): {forward_ensemble}')

In [ ]:
# Plot AUC vs. ensemble size
plt.figure(figsize=(8, 4))
plt.plot(forward_history['step'], forward_history['ensemble_auc'],
         marker='o', color='#003262', linewidth=2)
plt.axhline(auc_series.max(), color='gray', linestyle='--', label=f'Best single model ({auc_series.max():.3f})')
plt.xlabel('Ensemble size (number of models)')
plt.ylabel('Validation AUC')
plt.title('Forward Selection: AUC vs. Ensemble Size')
plt.legend()
plt.tight_layout()
plt.show()

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **Forward Selection is greedy - it cannot remove a previously added model. Modify `ensemble_forward` to include an early stopping criterion: stop adding models when the AUC improvement from the last addition is less than 0.001. How many models does this stopping rule select? Does stopping early hurt final test AUC?**

<br>

```python
def ensemble_forward_early_stop(preds_df, labels, n_models, min_improvement=0.001):
    remaining = list(preds_df.columns)
    selected = []
    prev_auc = 0.0
    # Add your stopping criterion here
    ...
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **BACKWARD** Elimination

<a id='Part_4_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 4.1: ALGORITHM AND IMPLEMENTATION

<br>

Backward Elimination starts with all models in the ensemble and greedily removes the one whose removal hurts AUC least (or improves it most). It is the mirror of Forward Selection.

**When to prefer Backward over Forward:**
- When models are highly correlated (many similar models that hurt diversity when all included)
- When you want to find a small, efficient ensemble from a large pool

**Computational cost:** Same $O(NK)$ as Forward Selection.

A known limitation: Backward Elimination can perform poorly when the starting ensemble of all $N$ models already has a lower AUC than the final ensemble. This happens when there are many "noise" models that hurt the ensemble before being pruned. Forward Selection avoids this by starting small and only adding models that help.

In [ ]:
def ensemble_backward(preds_df, labels, target_size):
    # Greedy backward elimination: remove the least helpful model at each step
    current = list(preds_df.columns)
    history = []

    while len(current) > target_size:
        baseline_auc = compute_ensemble_auc(preds_df, labels, current)
        best_auc_after_removal = -1
        model_to_remove = None

        for candidate in current:
            trial = [m for m in current if m != candidate]
            auc = compute_ensemble_auc(preds_df, labels, trial)
            if auc > best_auc_after_removal:
                best_auc_after_removal = auc
                model_to_remove = candidate

        current.remove(model_to_remove)
        history.append({
            'size': len(current),
            'removed': model_to_remove,
            'ensemble_auc': best_auc_after_removal
        })
        print(f'Removed {model_to_remove} -> Ensemble size {len(current)}, AUC = {best_auc_after_removal:.4f}')

    return pd.DataFrame(history), current

backward_history, backward_ensemble = ensemble_backward(val_preds, labels_val, target_size=3)
print(f'\nFinal Backward ensemble: {backward_ensemble}')

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **The `ensemble_backward` function above starts from all models and removes down to `target_size`. What happens to AUC at each removal step for your dataset? Does AUC monotonically increase as models are removed, or does it dip before recovering? Plot AUC vs. remaining ensemble size to see the trend.**

<br>

```python
# Plot the backward elimination history
plt.figure(figsize=(8, 4))
plt.plot(backward_history['size'], backward_history['ensemble_auc'], marker='o', color='#003262')
plt.xlabel('Remaining models')
plt.ylabel('Validation AUC')
plt.title('Backward Elimination: AUC vs. Ensemble Size')
plt.show()
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_5'></a>

<hr style="border: 2px solid#003262;" />

#### PART 5

## **RANDOM** Sampling and Comparison

<a id='Part_5_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 5.1: RANDOM ENSEMBLE SAMPLING

<br>

Random Ensemble Sampling draws a random subset of models (without replacement) and evaluates the resulting ensemble. Over many trials, it builds a distribution of AUC scores for ensembles of a given size.

**Why use random sampling?**
- It is a lower-effort baseline: no sequential logic, easy to parallelize
- It gives a distribution over possible ensembles, which is more informative than a single greedy path
- For small ensemble sizes and diverse model pools, random sampling can find good ensembles quickly

**Limitation:** It does not guarantee finding the best combination. For $N=15$ and $K=5$, there are $inom{15}{5} = 3003$ possible ensembles. Random sampling covers them stochastically; forward/backward selection takes a guided path through this space.

In [ ]:
def ensemble_random_sampling(preds_df, labels, ensemble_size, n_trials=50, random_state=42):
    # Sample random ensembles and return the best found
    rng = np.random.RandomState(random_state)
    cols = list(preds_df.columns)
    results = []
    for trial in range(n_trials):
        selected = list(rng.choice(cols, size=ensemble_size, replace=False))
        auc = compute_ensemble_auc(preds_df, labels, selected)
        results.append({'trial': trial, 'models': selected, 'auc': auc})
    results_df = pd.DataFrame(results)
    best_idx = results_df['auc'].idxmax()
    return results_df, results_df.loc[best_idx, 'models'], results_df.loc[best_idx, 'auc']

# Run random sampling at various ensemble sizes
random_results = {}
for k in [2, 3, 4, 5, 7, 10]:
    res, best_models, best_auc = ensemble_random_sampling(val_preds, labels_val, k, n_trials=50)
    random_results[k] = {'best_auc': best_auc, 'mean_auc': res['auc'].mean(), 'std': res['auc'].std()}
    print(f'k={k}: best AUC={best_auc:.4f}, mean={res["auc"].mean():.4f}, std={res["auc"].std():.4f}')

<a id='Part_5_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 5.2: COMPARING ALL THREE STRATEGIES

<br>

In [ ]:
# Compile results for comparison
comparison = []

# Forward selection at each size
for _, row in forward_history.iterrows():
    comparison.append({'strategy': 'Forward', 'size': int(row['step']), 'val_auc': row['ensemble_auc']})

# Random sampling best at each size
for k, v in random_results.items():
    comparison.append({'strategy': 'Random (best)', 'size': k, 'val_auc': v['best_auc']})
    comparison.append({'strategy': 'Random (mean)', 'size': k, 'val_auc': v['mean_auc']})

comp_df = pd.DataFrame(comparison)

plt.figure(figsize=(9, 5))
for strategy, grp in comp_df.groupby('strategy'):
    style = '-o' if strategy == 'Forward' else ('--s' if 'best' in strategy else ':^')
    plt.plot(grp['size'], grp['val_auc'], style, label=strategy, linewidth=2)

plt.axhline(auc_series.max(), color='black', linestyle=':', linewidth=1.5, label=f'Best single model')
plt.xlabel('Ensemble size')
plt.ylabel('Validation AUC')
plt.title('Ensemble Strategy Comparison')
plt.legend()
plt.tight_layout()
plt.show()

# Report final test AUC for the forward selection ensemble
print('\n--- Final Test AUC (forward ensemble) ---')
forward_test_auc = compute_ensemble_auc(test_preds, labels_test, forward_ensemble)
print(f'Forward ensemble ({len(forward_ensemble)} models): {forward_test_auc:.4f}')
best_single_test = max(roc_auc_score(labels_test, test_preds[col]) for col in test_preds.columns)
print(f'Best single model (test): {best_single_test:.4f}')
print(f'Ensemble gain over best single: {forward_test_auc - best_single_test:+.4f}')

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **The comparison above uses validation AUC to select the ensemble. Now compute test AUC for the best ensemble found by each strategy at size k=5. Which strategy produces the highest test AUC? Does the strategy that won on validation also win on test? What does any discrepancy tell you about overfitting the model selection process to the validation set?**

<br>

```python
# For each strategy, get the best ensemble of size 5
forward_5 = forward_ensemble[:5]
# get the best random ensemble at size 5 from the random_results above
random_best_5 = ...

# Compute test AUC for each
forward_test_5 = ...
random_test_5 = ...
print(f'Forward k=5 test AUC: {forward_test_5:.4f}')
print(f'Random k=5 test AUC: {random_test_5:.4f}')
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<hr style="border: 6px solid#003262;" />